# Reconstruction & Co-Registration

This workflow represents a comprehensive approach to quantitative mineral exploration that combines:

1. Plate Reconstruction: Understanding how geological terranes have moved through time

2. Spatial Analysis: Identifying favorable geological environments for ore formation

3. Temporal Analysis: Tracking how these environments evolved through geological history

4. Statistical Preparation: Creating datasets (deposit, unlabelled and target) suitable for predictive modeling

We begin by importing the required libraries:

In [ ]:
# =============================================================================
# IMPORT LIBRARIES AND DEPENDENCIES
# =============================================================================

from ipywidgets import interact
import os

# Geospatial and plotting libraries
import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

# Custom analysis modules
# NOTE: Ensure the 'lib/' folder is downloaded and in the same directory!
from lib.main import *

# Load configuration parameters (e.g. paths, model names)
from parameters import parameters

### Setup 

This section sets up the relevant parameters as specified in `'paramaters.py'`, namely:

- Temporal and spatial parameters for analysis.
- Plate model name.
- File and directory paths for input (including subduction zone characteristics, mineral deposit locations, target locations etc) and output data.

`NOTE`: You can change the analysis settings (like time range, resolution, and file locations) in `parameters.py`, which is in the same folder as this notebook.
It's structured as a dictionary; look for entries like 'plate_model_name', 'timespan', and 'grid_resolution' to adjust values as needed.

In [ ]:
# =============================================================================
# CONFIGURATION AND PARAMETERS
# =============================================================================

# Plate model name
plate_model_name = parameters["plate_model_name"]

# ------- Temporal configuration parameters -----
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"]
time_max = parameters["timespan"]["max"]
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

# ----- Spatial configuration parameters -----
# Distance in km for the buffer around subduction zones (used to find nearby deposits)
buffer_distance = parameters["buffer_distance"]
num_random = parameters["num_random"]   # Number of random deposits to generate
grid_resolution = parameters["grid_resolution"]

# ----- Data configuration parameters -----
# Directory paths for data sources and outputs
plate_model_dir = parameters["plate_model_dir"]
inputs_dir = parameters["inputs_dir"]
outputs_dir = parameters["outputs_dir"]
buffer_zones_dir = parameters["buffer_zones_dir"]

# Data file specifications
subduction_data_filename = parameters["subduction_data_filename"]
deposit_coords_filename = parameters["deposit_coords_filename"]
deposit_recon_coords_filename = parameters["deposit_recon_coords_filename"]
deposit_recon_coords_all_filename = parameters["deposit_recon_coords_all_filename"]
unlabelled_coords_filename = parameters["unlabelled_coords_filename"]
target_coords_filename = parameters["target_coords_filename"]
deposit_data_filename = parameters["deposit_data_filename"]
unlabelled_data_filename = parameters["unlabelled_data_filename"]
target_data_filename = parameters["target_data_filename"]

# Construct full file paths
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)
deposit_coords_filename = os.path.join(inputs_dir, deposit_coords_filename)
deposit_recon_coords_filename = os.path.join(outputs_dir, deposit_recon_coords_filename)
deposit_recon_coords_all_filename = os.path.join(outputs_dir, deposit_recon_coords_all_filename)
buffer_zones_dir = os.path.join(outputs_dir, buffer_zones_dir)
unlabelled_coords_filename = os.path.join(outputs_dir, unlabelled_coords_filename)
target_coords_filename = os.path.join(outputs_dir, target_coords_filename)
deposit_data_filename = os.path.join(outputs_dir, deposit_data_filename)
unlabelled_data_filename = os.path.join(outputs_dir, unlabelled_data_filename)
target_data_filename = os.path.join(outputs_dir, target_data_filename)

# Grid data directories for crustal properties
agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")        # Seafloor age grids
crusthick_dir = os.path.join(inputs_dir, "CrustalThickness") # Crustal thickness data

nprocs = 16

This section loads subduction and deposit location data, initializes the plate tectonic reconstruction model, and sets up a global map projection. It prepares the tools needed to reconstruct past plate positions, visualize subduction zones, and analyze the tectonic environment of mineral deposits through geological time.

In [ ]:
subduction_data = pd.read_csv(subduction_data_filename)     # Load subduction zone data
deposit_coords = pd.read_csv(deposit_coords_filename)       # Load deposit coordinates

# Load plate reconstruction model
plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

# Get topologies from the plate reconstruction model
gplot = get_plot_topologies(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
    plate_reconstruction=plate_model,
    filter_topologies=True,
)

# Using Mollweide projection for the map visualization
projection = ccrs.Mollweide(central_longitude=60)

### Buffer Zones

This section of code generates buffer zones around subduction trenches to define target areas. These represent back-arc basins where specific types of mineralization occur.

In [ ]:
# =============================================================================
# BUFFER ZONE CREATION
# =============================================================================

# Check if the buffer zones directory exists, if not, create it
if not os.path.isdir(buffer_zones_dir):
    run_create_buffer_zones(
        nprocs=nprocs,
        times=time_steps,
        plate_reconstruction=plate_model,
        output_dir=buffer_zones_dir,
        buffer_distance=buffer_distance,
        verbose=True,
        return_output=False,
    )

This section of code creates an interactive map to visualize the paleogeographic setting including seafloor age, plate boundaries, and target zones for mineralization. It shows:

- Seafloor age (color-coded from young ridges to old ocean floor)
- Continental crust (gray areas)
- Plate boundaries and mid-ocean ridges
- Subduction trenches with convergence indicators
- Target areas in back-arc basins (green zones)
- Plate motion vectors showing convergence directions

In [ ]:
# =============================================================================
# INTERACTIVE MAP VISUALIZATION
# =============================================================================

@interact
def show_map(time=time_steps):
    gplot.time = time

    # Load the seafloor age grid for the specified time
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5)),
    )

    # Plot seafloor age as background (blue=young, red=old)
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)

    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    
    # Show plate motion vectors (arrows indicating convergence directions)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Load and display buffer zones (target areas for mineralization)
    buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor='palegreen',
        edgecolor='none',
        alpha=0.7,
        zorder=4,
    )

    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)  # All plate boundaries
    gplot.plot_trenches(ax, color='k', zorder=6)                             # Subduction trenches
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', zorder=7)       # Convergence indicators
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    # Seafloor age colorbar
    cb = fig.colorbar(im, orientation='horizontal', shrink=0.4, pad=0.06, extend='max')
    cb.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.25))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
    
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Reconstruct Mineral Occurrences

The following two sections of code reconstruct mineral deposit locations backwards through geological time. It either loads precomputed paleoreconstructed deposit coordinates or generates them using plate tectonic reconstruction if they don't already exist.

In [ ]:
# Reconstruct deposits within buffer zones only (filtered dataset)
if os.path.isfile(deposit_recon_coords_filename):
    deposit_recon_coords = pd.read_csv(deposit_recon_coords_filename)
else:   
    # Reconstruct deposits if the file does not exist
    deposit_recon_coords = prepare_deposit_data(
        deposit_data=deposit_coords_filename,
        plate_reconstruction=plate_model,
        buffer_zones_dir=buffer_zones_dir,
        output_filename=deposit_recon_coords_filename,
        time_steps=time_steps,
        min_time=time_min,
        max_time=time_max,
        n_jobs=nprocs,
        verbose=True,
    )

In [ ]:
# Reconstruct all deposits (complete dataset for comparison)
if os.path.isfile(deposit_recon_coords_all_filename):
    deposit_recon_coords_all = pd.read_csv(deposit_recon_coords_all_filename)
else:
    deposit_recon_coords_all = partition_and_reconstruct(
        deposit_data=deposit_coords_filename,
        plate_reconstruction=plate_model,
        time_steps=time_steps,
        output_filename=deposit_recon_coords_all_filename,
        verbose=True,
    )

This section of code prepares a list of properties to plot for the interactive map. This map will help analyze the relationship between mineral deposits and subduction zone characteristics.

In [ ]:
# =============================================================================
# PREPARATION FOR VISUALIZATION
# =============================================================================

# Identify available subduction zone properties for visualization
subduction_data_columns = subduction_data.columns.tolist()

# Remove columns that are not relevant for plotting (e.g., coordinates, IDs)
features_plot = subduction_data_columns.copy()
features_plot.remove('lon')
features_plot.remove('lat')
features_plot.remove('age (Ma)')
features_plot.remove('subducting_plate_ID')
features_plot.remove('trench_plate_ID')

This section of code plots the interactive analysis of mineral deposits vs. subduction zone properties. It displays:

- Reconstructed mineral deposit locations (yellow circles, size = importance)
- Subduction zone characteristics (color-coded points along trenches)
- Seafloor age background for geological context
- Plate boundaries and convergence features

In [ ]:
# =============================================================================
# INTERACTIVE MAP VISUALIZATION
# =============================================================================

@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    # Get reconstructed deposit positions for this time
    deposit_recon_coords_t = deposit_recon_coords_all[[f'lon_{time}', f'lat_{time}', 'weight']]
    deposit_recon_coords_t = deposit_recon_coords_t.dropna()
    
    # Load seafloor age grid
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    features_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5)),
    )

    # Set up dual colorbars for feature and background
    cax_feat = fig.add_axes([0.1, 0.12, 0.35, 0.02])
    cax_bg = fig.add_axes([0.55, 0.12, 0.35, 0.02])
    
    # Plot seafloor age background
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Plot subduction zone features (color-coded by selected property)
    sc0 = ax.scatter(features_t['lon'], features_t['lat'], 50, marker='.',
                     c=features_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)
    
    # Add tectonic boundaries
    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)

    # Plot reconstructed mineral deposits (size proportional to importance/weight)
    sc1 = ax.scatter(
        deposit_recon_coords_t[f'lon_{time}'],
        deposit_recon_coords_t[f'lat_{time}'],
        transform=ccrs.PlateCarree(),
        marker='o',
        facecolor='yellow',
        edgecolor='black',
        s = [w * 10 for w in deposit_recon_coords_t['weight']],
        alpha=0.7,
        zorder=8
    )
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    # Colorbars for subduction feature and seafloor age
    cbar_feat = fig.colorbar(sc0, cax=cax_feat, orientation="horizontal")
    cbar_feat.set_label(feature, fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)

    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend='max')
    cbar_bg.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges'),
        Line2D([0], [0], marker='o', markerfacecolor='yellow', markeredgecolor='black', markersize=15, linestyle='None', label='Mineral Occurrence')
    ]

    # Add the custom legend to the plot
    legend = ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.15))
    
    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
    
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Unlabelled Samples

This code snippet is responsible for generating unlabelled sampling points, i.e. points that are not associated with known mineral deposits, but are used for comparison and training machine learning models (these serve as "negative" samples!) in a paleogeographic context.

These points are spatially distributed across buffer zones (regions of interest near reconstructed deposits or subduction zones), and are then paleoreconstructed to their historical positions using a tectonic plate model.

In [ ]:
# =============================================================================
# CREATE UNLABELLED SAMPLE POINTS
# =============================================================================

# Check if unlabelled coordinates file exists, if not, generate random points
if os.path.isfile(unlabelled_coords_filename):
    unlabelled_coords = pd.read_csv(unlabelled_coords_filename)
else:
    # Generate random points within buffer zones
    unlabelled_coords = generate_unlabelled_points(
        times=time_steps,
        input_dir=buffer_zones_dir,
        num=num_random, # Number of random points to generate
        threads=nprocs,
        seed=42,  # For reproducible random sampling
        plate_reconstruction=plate_model,
        verbose=True,
    )
    
    # Reconstruct these points through time
    unlabelled_coords = prepare_unlabelled_data(
        unlabelled_data=unlabelled_coords,
        plate_reconstruction=plate_model,
        output_filename=unlabelled_coords_filename,
        min_time=time_min,
        max_time=time_max,
        n_jobs=nprocs,
        verbose=True,
    )

This section of code plots a visualization of the randomly generated unlabelled sampling points within target areas. It shows:

- Target zones (green buffer areas around subduction trenches)
- Random sample points (cyan X markers)
- Geological context (seafloor age, plate boundaries)

These unlabelled points represent "unknown" or "non-deposit" areas and help identify what makes mineralized areas different from typical back-arc basin environments.

In [ ]:
# =============================================================================
# INTERACTIVE MAP VISUALIZATION
# =============================================================================

@interact
def show_map(time=time_steps):
    gplot.time = time    
    
    # Load seafloor age grid
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('silver', alpha=0.5))
    )

    # Background seafloor age
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)

    # Plot coastlines and plate motion vectors
    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Show target buffer zones
    buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    buffer_zones_t.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        facecolor='palegreen',
        edgecolor='none',
        alpha=0.7,
        zorder=4,
    )
    
    # Get unlabelled points for this time step
    unlabelled_coords_t = unlabelled_coords[unlabelled_coords["age (Ma)"] == time]

    # Add tectonic features
    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=7)

    # Plot unlabelled sample points
    ax.scatter(
        unlabelled_coords_t['lon'],
        unlabelled_coords_t['lat'],
        transform=ccrs.PlateCarree(),
        marker='X',
        facecolor='cyan',
        edgecolor='black',
        s=50,
        zorder=8
    )

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    # Colorbar for seafloor age
    cb = fig.colorbar(im, orientation='horizontal', shrink=0.4, pad=0.06, extend='max')
    cb.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)

    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.25))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Target Points

This snippet of code generates or loads a grid of target points that represent potential mineral exploration sites across geological time. These points represent locations where we want to predict mineralization potential.

In [ ]:
# =============================================================================
# CREATE TARGET POINTS
# =============================================================================

# Check if target coordinates file exists, if not, generate grid points
if os.path.isfile(target_coords_filename):
    target_coords = pd.read_csv(target_coords_filename)
    # Filter out rows with NaN coordinates
    target_coords = target_coords.dropna(subset=["present_lon", "present_lat"])
else:
    # Generate grid points for target areas based on buffer zones
    target_coords = generate_grid_points(
        times=time_steps,
        resolution=grid_resolution,
        polygons_dir=buffer_zones_dir,
        plate_reconstruction=plate_model,
        output_filename=target_coords_filename,
        n_jobs=nprocs,
        verbose=True,
    )

    target_coords = target_coords.dropna(subset=["present_lon", "present_lat"])

This section of code creates an interactive plot to understand the spatiotemporal context of target locations for mineral exploration. It shows:

- Dense grid of target points (red dots) for prediction
- Focus on areas where mineral potential will be assessed

This grid represents locations where we want to evaluate the likelihood of undiscovered mineral deposits based on the geological and geophysical characteristics learned from known deposits.

In [ ]:
# =============================================================================
# INTERACTIVE MAP VISUALIZATION
# =============================================================================

@interact
def show_map(time=time_steps):
    gplot.time = time    
    
    # Load seafloor age grid
    agegrid_filename =  f'seafloor_age_{time}Ma.nc'
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba('darkgray', alpha=0.5))
    )

    # Background seafloor age
    im = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)

    gplot.plot_coastlines(ax, facecolor='darkgray', edgecolor='none', zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # NOTE: Buffer zones commented out for cleaner visualization (?) of grid points

    # buffer_zones_t_filename = f'buffer_zones_{time}Ma.geojson'
    # buffer_zones_t_filename = os.path.join(buffer_zones_dir, buffer_zones_t_filename)
    # buffer_zones_t = gpd.read_file(buffer_zones_t_filename)

    # buffer_zones_t.plot(
    #     ax=ax,
    #     transform=ccrs.PlateCarree(),
    #     facecolor='palegreen',
    #     edgecolor='none',
    #     alpha=0.7,
    #     zorder=4,
    # )

    # Get target coordinates for this time step
    target_coords_t = target_coords[target_coords["age (Ma)"] == time]

    # Plot tectonic features
    gplot.plot_all_topologies(ax, color='dimgray', linewidth=1.5, zorder=5)

    # Plot systematic grid points (small red dots)
    ax.scatter(
        target_coords_t['lon'],
        target_coords_t['lat'],
        transform=ccrs.PlateCarree(),
        marker='.',
        c='red',
        s=1,
        alpha=0.5,
        zorder=6
    )
    
    # Add subduction features
    gplot.plot_trenches(ax, color='k', alpha=0.3, zorder=7)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color='k', alpha=0.3, zorder=8)

    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=9)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
    
    # Colorbar for seafloor age
    cb = fig.colorbar(im, orientation='horizontal', shrink=0.4, pad=0.06, extend='max')
    cb.set_label("Seafloor Age (Ma)", fontsize=16, labelpad=10)
    cb.set_ticks([0, 50, 100, 150, 200])
    cb.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor='darkgray', edgecolor='none', label='Continental Crust'),
        # Patch(facecolor='palegreen', edgecolor='none', alpha=0.7, label='Target Areas in\nBack-Arc Basins'),
        Line2D([0], [0], color='dimgray', lw=2, label='Mid-Ocean Ridges'),
        Line2D([0], [0], marker='.', markerfacecolor='red', markeredgecolor='none', markersize=10, linestyle='None', label='Target Points')
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc='lower left', bbox_to_anchor=(0, -0.25))

    ax.set_title(f'{time} Ma', fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …

### Coregistration

This code block performs co-registration: the process of attaching meaningful geodynamic context (e.g., subduction information, crustal thickness) to different sets of geological points. Specifically:

1. `deposit_data`: Known mineral deposits, used as positive training data.

2. `unlabelled_data`: Background points, used as negative or unlabelled training data.

3. `target_data`: Points of interest, used as prediction targets.

After co-registration, each point (whether deposit, background, or target) has a complete feature vector, including:

- Proximity to subduction zones and plate motion data

- Crustal thickness at the corresponding geological time

In [14]:
# =============================================================================
# CO-REGISTRATION
# =============================================================================

# MINERAL DEPOSITS: Extract features for known mineralization sites
if not os.path.isfile(deposit_data_filename):
    # Adds tectonic context (subduction zone properties)
    deposit_data = run_coregister_point_data(
        point_data=deposit_recon_coords,
        subduction_data=subduction_data,
        n_jobs=nprocs,
        verbose=True,
    )
    
    # Adds crustal thickness data (physical context)
    deposit_data = run_coregister_crustal_thickness(
        point_data=deposit_data,
        input_dir=crusthick_dir,
        output_filename=deposit_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )

# UNLABELLED POINTS: Extract features for random background points
if not os.path.isfile(unlabelled_data_filename):
    unlabelled_data = run_coregister_point_data(
        point_data=unlabelled_coords,
        subduction_data=subduction_data,
        output_filename=unlabelled_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )
    
    unlabelled_data = run_coregister_crustal_thickness(
        point_data=unlabelled_data,
        input_dir=crusthick_dir,
        output_filename=unlabelled_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )

# TARGET POINTS: Extract features for points of interest
if not os.path.isfile(target_data_filename):
    target_data = run_coregister_point_data(
        point_data=target_coords,
        subduction_data=subduction_data,
        output_filename=target_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )
    
    target_data = run_coregister_crustal_thickness(
        point_data=target_data,
        input_dir=crusthick_dir,
        output_filename=target_data_filename,
        n_jobs=nprocs,
        verbose=True,
    )